In [1]:
import os

from loadData import data_pipe
os.environ['CUDA_VISIBLE_DEVICES'] = ','.join(map(str, [1]))
print('using GPU %s' % ','.join(map(str, [1])))

import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import SequentialLR, StepLR, LambdaLR
from thop import profile, clever_format

import csv
import time
import json
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
# plt.rc('font',family='Times New Roman') 

from option import opt
from loadData import data_pipe
from loadData.split_data import HyperX2, HyperX3
from loadData.dataAugmentation import DataAugmentation

from models import vision_transformer, CNNBase, vision_transformer_dino
from models import automaticWeightedLoss, modules
from models.MS2CANet import pymodel
# from models.Morphs import morphFormer2
from utils import trainer, tester, infoNCE, tools, focalLoss
from utils import mutual_info, distillation, CELoss, visulization

# from utils import orthogonalLoss

using GPU 1


/home/icclab/Documents/lqw/Multimodal_Classification/MoEIF/models/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/icclab/Documents/lqw/Multimodal_Classification/MoEIF/models/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/icclab/Documents/lqw/Multimodal_Classification/MoEIF/models/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


In [2]:
args = opt.get_args()
args.dataset_name = "Houston_2013"
# args.dataset_name = "Houston_2018"
# args.dataset_name = "Berlin"

# args.backbone = "MiViT"
args.backbone = "resNet2"
# args.backbone = "resNet4"
# args.backbone = "morphFormer2"
# args.backbone = "vit_dino_s"
# args.backbone = "vit_dino_b"
# args.local_crops_number = 0

# args.split_type = "disjoint"
args.split_type = "ratio"

print("args.backbone", args.backbone)
# print("args.randomCrop", args.randomCrop)

args.backbone resNet2


In [3]:
args.epochs = 50
args.batch_size = 64
args.patch_size = 13
args.randomCrop = 11

# args.patch_size = 14
# args.randomCrop = 12

# args.patch_size = 6
# args.randomCrop = 4
args.pca = True
args.components = 15
# args.pca = False
# args.components = 0

args.lb_smooth = 0.1
args.learning_rate = 0.001
# args.learning_rate = 0.0005
args.weight_decay = 1e-3
args.lambda_contra = 0.1
args.lambda_orth = 0.1
args.lambda_kl = 0
args.lambda_super = 1
args.schedule = True
args.step_size = 30
args.gamma = 0.7

In [4]:
args.print_data_info = False
args.data_info_start = 1
args.show_gt = False
args.remove_zero_labels = True
args.train_ratio = 1

# create dataloader
img1, img2, train_gt, val_gt, test_gt, data_gt, GT = data_pipe.get_data(args)
transform = DataAugmentation(args)

args.contrastive = True
# args.mix = True
contrastive_dataset = HyperX2(img1, data2=img2, gt=data_gt, transform=transform, args=args)
contrastive_loader = DataLoader(contrastive_dataset, batch_size=args.batch_size, shuffle=True, drop_last=True)


# data_pipe.set_deterministic(seed = 666)
args.print_data_info = True
args.data_info_start = 1
args.show_gt = False
args.remove_zero_labels = True
args.train_ratio = 0.9

# create dataloader
img1, img2, train_gt, val_gt, test_gt, data_gt, GT = data_pipe.get_data(args)
band1 = img1.shape[2]
band2 = img2.shape[2]
args.min = str(np.min(img1))
args.max = str(np.max(img1))
print("img1", img1.shape, "img2", img2.shape, band1, band2, args.min, args.max)
print("train_gt", train_gt.shape, \
		"test_gt", test_gt.shape, \
		"data_gt", data_gt.shape, \
		"GT", GT.shape)
transform = DataAugmentation(args)


# 不拆分高光谱图像的图像处理
# args.mix = True
# contrastive_dataset = HyperX2(img1, data2=img2, gt=data_gt, transform=transform, args=args)
args.contrastive = False
# args.mix = False
train_dataset = HyperX2(img1, data2=img2, gt=train_gt, transform=None, args=args)
val_dataset = HyperX2(img1, data2=img2, gt=val_gt, transform=None, args=args)
test_dataset = HyperX2(img1, data2=img2, gt=test_gt, transform=None, args=args)

# 用于 focalloss
train_gt_pure = train_gt[train_gt > 0] - 1
test_gt_pure = test_gt[test_gt > 0] - 1
loss_weight = focalLoss.loss_weight_calculation(test_gt_pure)

# contrastive_loader = DataLoader(contrastive_dataset, batch_size=args.batch_size, shuffle=True, drop_last=True)
train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)

class_num = np.max(train_gt)
print(class_num, train_gt.shape, len(train_loader.dataset))

pca is used
pca is used
print_data_info : ---->
class 1 	 178 	 20 	 1053
class 2 	 171 	 19 	 1064
class 3 	 173 	 19 	 505
class 4 	 169 	 19 	 1056
class 5 	 167 	 19 	 1056
class 6 	 164 	 18 	 143
class 7 	 176 	 20 	 1072
class 8 	 172 	 19 	 1053
class 9 	 174 	 19 	 1059
class 10 	 172 	 19 	 1036
class 11 	 163 	 18 	 1054
class 12 	 173 	 19 	 1041
class 13 	 166 	 18 	 285
class 14 	 163 	 18 	 247
class 15 	 168 	 19 	 473
total     	 2549 	 283 	 12197
img1 (363, 1919, 15) img2 (363, 1919, 1) 15 1 -61.105827 69.57207
train_gt (363, 1919) test_gt (363, 1919) data_gt (363, 1919) GT (349, 1905)
15 (363, 1919) 2549


In [5]:
for x11, x21, x12, x22, y in contrastive_loader:
    print("x11.shape, x12.shape", x11.shape, x12.shape, len(x11))
    print("x21.shape, x22.shape", x21.shape, x22.shape, len(x21))

    # fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    # for i in range(9):
    #     ax = axes[i // 3, i % 3]
    #     img = x11[i].permute(1, 2, 0)[:, :, :3]  # 转为HWC格式，并取前3通道
    #     img = (img - img.min()) / (img.max() - img.min() + 1e-6)  # 归一化以避免显示错误
    #     ax.imshow(img)
    #     ax.axis('off')
        
    # plt.tight_layout(pad=0)  # 去除额外空白边距
    # plt.show()
    break

for x11, x21, y in train_loader:
    print("x11.shape", x11.shape)
    print("x21.shape", x21.shape)
    print("y.shape", y.shape, y.type())
    break

x11.shape, x12.shape torch.Size([64, 15, 11, 11]) torch.Size([64, 15, 11, 11]) 64
x21.shape, x22.shape torch.Size([64, 1, 11, 11]) torch.Size([64, 1, 11, 11]) 64
x11.shape torch.Size([64, 15, 13, 13])
x21.shape torch.Size([64, 1, 13, 13])
y.shape torch.Size([64]) torch.LongTensor


# Path

In [6]:
# 加载已有权重路径
# args.result_dir = "/home/icclab/Documents/lqw/Multimodal_Classification/KnowCLPlus/result/06-22-10-02-resNet2_Houston_2013"

args.result_dir = os.path.join("/home/icclab/Documents/lqw/Multimodal_Classification/MoEIF/result",
                    datetime.now().strftime("%m-%d-%H-%M-" + args.backbone + "_" + args.dataset_name))
print(args.result_dir)

if not os.path.exists(args.result_dir):
    os.mkdir(args.result_dir)
with open(args.result_dir + '/args.json', 'w') as fid:
    json.dump(args.__dict__, fid, indent=2)

/home/icclab/Documents/lqw/Multimodal_Classification/MoEIF/result/08-02-16-16-resNet2_Houston_2013


# model

In [7]:
if args.backbone == "resNet2":
    encoder = CNNBase.Model_base(channell=band1, channel2=band2).to(args.device)
    args.feature_dim = 512
    # args.feature_dim = 2048


# elif args.backbone == "resNet4":
#     encoder = resNet4.Model_base(channell=band1, channel2=band2).to(args.device)
#     args.feature_dim = 512
#     # args.feature_dim = 2048

elif args.backbone == "vit_dino_s":
    selected_layers = [0, 2, 5, 7]  # 从 dinov2 中选择的层
    encoder = vision_transformer_dino.Vit_base(band1, band2, args.randomCrop, selected_layers=selected_layers).to(args.device)
    args.feature_dim = 384

# elif args.backbone == "vit_dino_b":
#     selected_layers = [0, 2, 5, 7]  # 从 dinov2 中选择的层
#     encoder = vision_transformer_dino.vit_selected_base(band1, band2, args.randomCrop, selected_layers=selected_layers).to(args.device)
#     args.feature_dim = 768 

# elif args.backbone == "morphFormer2":
#     args.FM = 16
#     encoder = morphFormer2.MorphFormer(args.FM, band1, band2, class_num, args.randomCrop).to(args.device)
#     args.feature_dim = 64


super_head = modules.FDGCHead(args.feature_dim, class_num=class_num).to(args.device)
contra_head = modules.DINOHead(args.feature_dim).to(args.device)
awl = automaticWeightedLoss.AutomaticWeightedLoss(3).to(args.device)
params = list(super_head.parameters()) + list(encoder.parameters()) + list(contra_head.parameters())

optimizer = torch.optim.AdamW(params, lr=args.learning_rate, weight_decay=args.weight_decay)
warmup = LambdaLR(optimizer, lr_lambda=lambda e: min(1.0, e / 5.0))
decay = StepLR(optimizer, step_size=args.step_size, gamma=args.gamma)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=args.step_size, gamma=args.gamma)  # 学习太快
scheduler = SequentialLR(optimizer, schedulers=[warmup, decay], milestones=[5])


# criterion0 = torch.nn.CrossEntropyLoss().to(args.device)
criterion1 = CELoss.LabelSmoothSoftmaxCEV1(lb_smooth=args.lb_smooth).to(args.device)
criterion2 = infoNCE.InfoNCE().to(args.device)
criterion3 = infoNCE.NT_xent_loss_W_EN().to(args.device)

# criterion4 = distillation.KL_loss().to(args.device)
# criterion5 = mutual_info.Mutual_info_cnn(args.feature_dim, args.feature_dim).to(args.device)
# criterion6 = orthogonalLoss.OrthogonalLoss(version='dot', sample_size=20).to(args.device)
loss_weight = loss_weight.to(args.device)
criterion7 = focalLoss.FocalLoss(loss_weight, gamma=2, alpha=None).to(args.device)

In [8]:
# input_hsi = torch.randn(1, band1, args.patch_size, args.patch_size).cuda()
# input_lidar = torch.randn(1, band2, args.patch_size, args.patch_size).cuda()
# # flops, params = profile(model, inputs=(input_hsi, input_lidar))

# flops, params = profile(encoder, inputs=(input_hsi, input_lidar))
# flops, params = clever_format([flops, params])
# print('# Model Params: {} FLOPs: {}'.format(params, flops))


# # contra_head
# flops, params = profile(contra_head, inputs=(torch.randn(1, args.feature_dim).cuda(),
# 									torch.randn(1, args.feature_dim).cuda(),
# 									torch.randn(1, args.feature_dim).cuda(),))
# flops, params = clever_format([flops, params])
# print('# Model Params: {} FLOPs: {}'.format(params, flops))


# # super_head
# flops, params = profile(super_head, inputs=(torch.randn(1, args.feature_dim).cuda(),))
# flops, params = clever_format([flops, params])
# print('# Model Params: {} FLOPs: {}'.format(params, flops))

# 训练前加载权重

In [9]:
# args.result_dir = "/home/icclab/Documents/lqw/Multimodal_Classification/KnowCLPlus/result_PAC/07-10-22-25-resNet2_Berlin"
from pathlib import Path
args.resume = Path(os.path.join(args.result_dir, "test_loss.pth"))

if args.resume.is_file():
    checkpoint = torch.load(args.resume)
    encoder.load_state_dict(checkpoint['model'], strict=False)
    super_head.load_state_dict(checkpoint['super_head'], strict=False)
    contra_head.load_state_dict(checkpoint['contra_head'], strict=False)
    epoch = checkpoint['epoch'] + 1
    print('Loaded from: {} epoch {}'.format(args.resume, epoch))
else:
    epoch_start = 0

# 训练

In [ ]:
best_loss = 999
best_acc = 0
train_losses = []
test_losses = []
loss_contras = []
loss_orths = []
loss_supers = []
train_accuracies = []
test_accuracies = []
start_time = time.time()
for epoch in range(epoch_start, args.epochs):
###################################################################->>>>>>>>>

    if args.backbone == "resNet2" or args.backbone == "resNet4":
        train_loss, loss_contra, loss_orth, loss_super, loss_dist, train_accuracy, train_time = \
                    trainer.train(encoder, contra_head, super_head, awl, criterion1, \
                                criterion2, criterion3, contrastive_loader, \
                                train_loader, optimizer, args)
        if epoch % args.log_interval == 0:
            test_loss, test_preds, targets, test_accuracy, test_time = \
                    tester.test_resNet2(encoder, super_head, criterion1, val_loader, args) 

    elif args.backbone == "vit_dino_s" or args.backbone == "vit_dino_b":
        train_loss, loss_contra, loss_orth, loss_super, loss_dist, train_accuracy, train_time = \
                    trainer.train(encoder, contra_head, super_head, awl, criterion1, \
                                criterion2, contrastive_loader, \
                                train_loader, optimizer, args)
        if epoch % args.log_interval == 0:
            test_loss, test_preds, targets, test_accuracy, test_time = \
                    tester.test_resNet2(encoder, super_head, criterion2, val_loader, args) 
            
    elif args.backbone == "morphFormer2":
        train_loss, loss_contra, loss_orth, loss_super, loss_dist, train_accuracy, train_time = \
                    trainer.train_morphFormer2(encoder, contra_head, super_head, awl, criterion1, \
                                criterion2, contrastive_loader, \
                                train_loader, optimizer, args)
        if epoch % args.log_interval == 0:
            test_loss, test_preds, targets, test_accuracy, test_time = \
                    tester.test_morphFormer2(encoder, super_head, criterion2, val_loader, args)
    else:
        raise NotImplementedError("No models")         
        
    print('Train Epoch: [{}/{}] Loss: {:.4f} lco: {:.4f} lor: {:.4f} lsu: {:.4f} ldt: {:.4f} TrainAcc: {:.2f} TestAcc: {:.2f} TIME: {:.4f}'.format(\
                            epoch, args.epochs, 
                            train_loss, 
                            loss_contra, 
                            loss_orth, 
                            loss_super, 
                            loss_dist,
                            train_accuracy, 
                            test_accuracy, 
                            train_time
                            ))

    with open(os.path.join(args.result_dir, "log.csv"), 'a+', encoding='gbk') as f:
        row=[["epoch", epoch,
            "loss", train_loss,
            "test_loss", test_loss,
            "loss_contra", loss_contra,
            "loss_super", loss_super,
            "loss_dist", loss_dist,
            "train_accuracy", train_accuracy,
            "test_accuracy", test_accuracy,
            "train_time", train_time,
            "test_time", test_time,
            '\n']]
        write=csv.writer(f)
        for i in range(len(row)):
            write.writerow(row[i])

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    loss_contras.append(loss_contra)
    loss_orths.append(loss_orth)
    loss_supers.append(loss_super)
    train_accuracies.append(train_accuracy)
    test_accuracies.append(test_accuracy)

    scheduler.step()

    best_loss, best_acc = tools.save_weights(train_loss, test_loss, best_loss, best_acc, \
                                    test_accuracy, epoch, encoder, contra_head, super_head, optimizer, args)

Total_train_time = time.time() - start_time
torch.save({
        "epoch": epoch,
        "base": encoder.state_dict(),
        "contra_head": contra_head.state_dict(),
        "super_head": super_head.state_dict(),
        "optimizer": optimizer.state_dict()}, 
os.path.join(args.result_dir, "model_last.pth"))

Train Epoch: [0/50] Loss: 4.3981 lco: 1.6259 lor: 0.0000 lsu: 2.7722 ldt: 0.0000 TrainAcc: 6.75 TestAcc: 5.65 TIME: 20.9600
Train Epoch: [1/50] Loss: 2.3910 lco: 1.2644 lor: 0.0000 lsu: 1.1266 ldt: 0.0000 TrainAcc: 54.96 TestAcc: 76.33 TIME: 19.9100
Train Epoch: [2/50] Loss: 1.8512 lco: 1.0471 lor: 0.0000 lsu: 0.8042 ldt: 0.0000 TrainAcc: 91.53 TestAcc: 96.11 TIME: 20.2600
Train Epoch: [3/50] Loss: 1.6706 lco: 1.0462 lor: 0.0000 lsu: 0.6244 ldt: 0.0000 TrainAcc: 98.31 TestAcc: 98.94 TIME: 20.2500
Train Epoch: [4/50] Loss: 1.6362 lco: 0.9760 lor: 0.0000 lsu: 0.6603 ldt: 0.0000 TrainAcc: 98.43 TestAcc: 97.53 TIME: 20.2400


/home/icclab/miniconda3/envs/leo/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:198: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Train Epoch: [5/50] Loss: 1.5781 lco: 0.9429 lor: 0.0000 lsu: 0.6352 ldt: 0.0000 TrainAcc: 99.41 TestAcc: 98.94 TIME: 20.6100
Train Epoch: [6/50] Loss: 1.5739 lco: 0.9370 lor: 0.0000 lsu: 0.6368 ldt: 0.0000 TrainAcc: 99.69 TestAcc: 99.65 TIME: 20.6100
Train Epoch: [7/50] Loss: 1.4982 lco: 0.8904 lor: 0.0000 lsu: 0.6078 ldt: 0.0000 TrainAcc: 99.88 TestAcc: 99.29 TIME: 20.2300
Train Epoch: [8/50] Loss: 1.5022 lco: 0.9008 lor: 0.0000 lsu: 0.6014 ldt: 0.0000 TrainAcc: 99.76 TestAcc: 99.65 TIME: 20.1600
Train Epoch: [9/50] Loss: 1.4666 lco: 0.8648 lor: 0.0000 lsu: 0.6018 ldt: 0.0000 TrainAcc: 100.00 TestAcc: 99.29 TIME: 20.1500
Train Epoch: [10/50] Loss: 1.4436 lco: 0.8542 lor: 0.0000 lsu: 0.5894 ldt: 0.0000 TrainAcc: 99.96 TestAcc: 98.94 TIME: 20.2500
Train Epoch: [11/50] Loss: 1.4679 lco: 0.8795 lor: 0.0000 lsu: 0.5883 ldt: 0.0000 TrainAcc: 100.00 TestAcc: 98.94 TIME: 20.6100
Train Epoch: [12/50] Loss: 1.4243 lco: 0.8363 lor: 0.0000 lsu: 0.5880 ldt: 0.0000 TrainAcc: 99.96 TestAcc: 98.94 T

# 验证精度

In [ ]:
args.resume = os.path.join(args.result_dir, "test_loss.pth")
if args.resume != '':
    checkpoint = torch.load(args.resume)
    encoder.load_state_dict(checkpoint['model'], strict=False)
    super_head.load_state_dict(checkpoint['super_head'], strict=False)
    contra_head.load_state_dict(checkpoint['contra_head'], strict=False)
    epoch = checkpoint['epoch'] + 1
    print('Loaded from: {} epoch {}'.format(args.resume, epoch))
else:
    epoch_start = 0

# linear 精度
if args.backbone == "resNet2" or args.backbone == "resNet4":
    test_loss, test_preds, targets, test_acc, test_time = \
        tester.test_resNet2(encoder, super_head, criterion1, test_loader, args) 
# elif args.backbone == "vit_dino_s" or args.backbone == "vit_dino_b":
#     test_loss, test_preds, targets, test_acc, test_time = \
#         tester.test_resNet2(encoder, super_head, criterion2, test_loader, args) 
# elif args.backbone == "morphFormer2":
#     test_loss, test_preds, targets, test_acc, test_time = \
#         tester.test_morphFormer2(encoder, super_head, criterion2, test_loader, args)
else:
    raise NotImplementedError("No models")
classification, kappa = tester.get_results(test_preds, targets)
print(classification)


with open(os.path.join(args.result_dir, "log_final.csv"), 'a+', encoding='gbk') as f:
    row=[["\nLinear Train",
        "\nepoch", epoch, 
        "\nclassification\n", classification,
        "\nkappa", kappa,
        "\nTest_time", round(test_time, 2),
        "\nTrian_time", round(Total_train_time, 2),
        "\n"
        ]]
    write=csv.writer(f)
    for i in range(len(row)):
        write.writerow(row[i])

# 绘图

In [ ]:
# for i, j, y in data_loader:
# 	print("i.shape, j.shape", i.shape, j.shape)
# 	break

In [ ]:
# args.resume = os.path.join(args.result_dir, "test_loss.pth")
# if args.resume != '':
#     checkpoint = torch.load(args.resume)
#     encoder.load_state_dict(checkpoint['model'], strict=False)
#     super_head.load_state_dict(checkpoint['super_head'], strict=False)
#     contra_head.load_state_dict(checkpoint['contra_head'], strict=False)
#     epoch = checkpoint['epoch'] + 1
#     print('Loaded from: {} epoch {}'.format(args.resume, epoch))
# else:
#     epoch_start = 0

# args.print_data_info = False
# args.data_info_start = 1
# args.show_gt = False
# args.remove_zero_labels = False
# args.train_ratio = 1

# # create dataloader
# img1, img2, train_gt, val_gt, test_gt, data_gt, groundTruth = data_pipe.get_data(args)
# # transform = DataAugmentation(args)

# data_dataset = HyperX2(img1, data2=img2, gt=data_gt, transform=None, args=args)
# data_loader = DataLoader(data_dataset, batch_size=args.batch_size, shuffle=False, drop_last=False)

# visulization.test_visulization(encoder, super_head, data_loader, args, groundTruth) 

# 画 loss 曲线

In [ ]:
# args.plot_loss_curve = True
# if args.plot_loss_curve:
#     # fig = plt.figure()
#     fig, ax1 = plt.subplots()

#     # 绘制第一个数据，使用左侧y轴
#     ax1.plot(range(args.epochs), train_losses, 'blue', label="train_loss")
#     ax1.plot(range(args.epochs), test_losses, 'gray', label="test_loss")
#     ax1.set_xlabel('X')
#     ax1.set_ylabel('loss', color='b')
#     ax1.tick_params(axis='y', labelcolor='b')

#     # 创建第二个坐标轴，共享x轴但y轴在右侧
#     ax2 = ax1.twinx()
#     ax2.plot(range(args.epochs), train_accuracies, 'red', label="train_accuracy")
#     ax2.plot(range(args.epochs), test_accuracies, 'pink', label="test_accuracy")
#     ax2.set_ylabel('accuracy', color='r')
#     ax2.tick_params(axis='y', labelcolor='r')


#     fig.tight_layout()  # 自动调整布局以避免重叠

#     # # 获取两个坐标轴上的线条和标签
#     lines_1, labels_1 = ax1.get_legend_handles_labels()
#     lines_2, labels_2 = ax2.get_legend_handles_labels()
#     # ax1.legend(lines_1, labels_1, loc=5)
#     ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc=5)
#     # plt.legend(['train_losses', 'train_accuracies', 'test_accuracies'], loc='upper right')
#     # plt.xlabel('number of training examples seen')
#     # plt.ylabel('Accuracy')
#     plt.savefig(os.path.join(args.result_dir, "learning_curve.png"), dpi=200)

# TSNE

In [ ]:
# args.resume = os.path.join(args.result_dir, "test_loss.pth")
# if args.resume != '':
#     checkpoint = torch.load(args.resume)
#     encoder.load_state_dict(checkpoint['model'], strict=False)
#     super_head.load_state_dict(checkpoint['super_head'], strict=False)
#     contra_head.load_state_dict(checkpoint['contra_head'], strict=False)
#     epoch = checkpoint['epoch'] + 1
#     print('Loaded from: {} epoch {}'.format(args.resume, epoch))
# else:
#     epoch_start = 0

# args.print_data_info = False
# args.data_info_start = 1
# args.show_gt = False
# args.remove_zero_labels = True
# args.train_ratio = 0.9

# # create dataloader
# img1, img2, train_gt, val_gt, test_gt, data_gt, groundTruth = data_pipe.get_data(args)
# # transform = DataAugmentation(args)

# data_dataset = HyperX2(img1, data2=img2, gt=test_gt, transform=None, args=args)
# data_loader = DataLoader(data_dataset, batch_size=args.batch_size, shuffle=False, drop_last=False)

In [ ]:
# with torch.no_grad():
#     for idx_o, (S_1, S_2, target) in enumerate(data_loader):

#         if idx_o >= 200:
#             break

#         # target = target - 1
#         S_1 = S_1.to(args.device)
#         S_2 = S_2.to(args.device)
#         # target = target.to(args.device)
            
#         s_base1, s_base2, s_fuse = encoder(S_1, S_2)
#         output = super_head(s_fuse)

#         # ✅ 只取前 N 个样本
#         output = output[:100].cpu().numpy()
#         target = target[:100]

#         for idx, _ in enumerate(output):
#             # print(output.shape)
#             if idx == 0 and idx_o == 0:
#                 list_output = output[idx]
#                 list_target = target[idx]
#             # print(list_output.shape, list_target.shape)

#             else:
#                 list_output = np.vstack((list_output, output[idx]))
#                 list_target = np.append(list_target, target[idx])

In [ ]:
# tsne = TSNE()

# out = tsne.fit_transform(list_output)
# fig, ax = plt.subplots()
# ax.set_axis_off()
# ax.xaxis.set_visible(False)
# ax.yaxis.set_visible(False)
# # fig.set_size_inches(label.shape[1] * scale / dpi, label.shape[0] * scale / dpi)
# plt.gca().xaxis.set_major_locator(plt.NullLocator())
# plt.gca().yaxis.set_major_locator(plt.NullLocator())
# plt.subplots_adjust(top=1, bottom=0, right=1, left=0, hspace=0, wspace=0)

# # label = ["Broccoli-weeds-1","Broccoli-weeds-2","Fallow","Fallow-rough-plow",
# #          "Fallow-smooth","Stubble","Celery","Grapes-untrained",
# #          "Soil-senesced-develop","Corn-weeds","Lettuce-4wk","Lettuce-5wk",
# #          "Lettuce-6wk","Lettuce-7wk","Vinyard-untrained",
# #          "Vinyard-vertical-trellis"]
# for i in range(class_num):
#     indices = list_target == i
#     x, y = out[indices].T
#     # plt.scatter(x, y, s=5, label=label[i])
#     plt.scatter(x, y, s=10, label=str(i+1))
# plt.legend(loc=2, bbox_to_anchor=(1.05,1.0), borderaxespad = 0., fontsize=16, markerscale=2.5)  # ✅ 放大图例中的点)
# plt.savefig(os.path.join(args.result_dir, 'TSNE' + '.png'), dpi=400, bbox_inches="tight")